# 📊 Merged Netflix Dataset (netflix.csv) - Aira

This notebook was created to understand, validate, and profile the merged Netflix dataset before moving into more focused exploratory analysis and dashboard storytelling.

----

# 💡 Early Insights Found

Based on the notebook flow, the main insights are:

- The merged dataset is large enough for meaningful exploration.
- It includes a broad multi-country scope across many weeks and titles.
- The file supports both country-level and global title-level analysis.
- Some fields have null values, which is expected in merged datasets and needs to be considered during later filtering and KPI design.
- The dataset can already support title-based ranking logic using: appearances, average rank, longevity
- The notebook begins shifting from general profiling into show-title-centric analysis, which is useful for later Power BI storytelling.

| KPI / Metric    | Meaning                    | What it helps measure     |
| --------------- | -------------------------- | ------------------------- |
| **appearances** | How often a title shows up | visibility / consistency  |
| **avg_rank**    | Average ranking position   | performance quality       |
| **longevity**   | Long-term Top 10 presence  | staying power / lifecycle |

----

In [8]:
import pandas as pd
import duckdb
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# --- 1. Resolve project root (robust) ---
BASE_DIR = Path().resolve()
while not (BASE_DIR / "data").exists():
    BASE_DIR = BASE_DIR.parent

# --- 2. Centralized paths ---
DATA_RAW = BASE_DIR / "data" / "raw"
DATA_PROCESSED = BASE_DIR / "data" / "processed"
DB_PATH = BASE_DIR / "database" / "netflix.duckdb"

# --- 3. Connect to DuckDB ---
con = duckdb.connect(DB_PATH)

# --- 4. Load dataset ---
df = pd.read_csv(DATA_PROCESSED / "netflix.csv")

# --- 5. Register + persist table ---
duckdb.register("netflix_df", df)

duckdb.sql("""
CREATE OR REPLACE TABLE netflix AS
SELECT * FROM netflix_df
""")

# --- 6. Quick inspection ---
print(df.shape)
print(df.columns)
df.head()

(458338, 15)
Index(['Unnamed: 0', 'country_name', 'country_iso2', 'week',
       'country_category', 'country_weekly_rank', 'show_title', 'season_title',
       'country_cumulative_weeks_in_top_10', 'global_category',
       'global_weekly_rank', 'global_weekly_hours_viewed', 'runtime',
       'global_weekly_views', 'global_cumulative_weeks_in_top_10'],
      dtype='object')


,Unnamed: 0,country_name,country_iso2,week,country_category,country_weekly_rank,show_title,season_title,country_cumulative_weeks_in_top_10,global_category,global_weekly_rank,global_weekly_hours_viewed,runtime,global_weekly_views,global_cumulative_weeks_in_top_10
0,0,Argentina,AR,2026-03-15,Films,1,War Machine,NaN,2,Films (English),1.0,80600000.0,109.002,44400000.0,2.0
1,1,Argentina,AR,2026-03-15,Films,2,Strangers in the Park,NaN,2,Films (Non-English),10.0,1600000.0,115.998,800000.0,2.0
2,2,Argentina,AR,2026-03-15,Films,3,Joker: Folie à Deux,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN
3,3,Argentina,AR,2026-03-15,Films,4,Trolls Band Together,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN
4,4,Argentina,AR,2026-03-15,Films,5,Double Jeopardy,NaN,1,Films (English),5.0,6200000.0,106.002,3500000.0,1.0


In [9]:
df.info()
df.describe(include="all")
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 458338 entries, 0 to 458337
Data columns (total 15 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   Unnamed: 0                          458338 non-null  int64  
 1   country_name                        458338 non-null  object 
 2   country_iso2                        458338 non-null  object 
 3   week                                458338 non-null  object 
 4   country_category                    458338 non-null  object 
 5   country_weekly_rank                 458338 non-null  int64  
 6   show_title                          458338 non-null  object 
 7   season_title                        223904 non-null  object 
 8   country_cumulative_weeks_in_top_10  458338 non-null  int64  
 9   global_category                     306075 non-null  object 
 10  global_weekly_rank                  306075 non-null  float64
 11  global_weekly_hours_viewed

Unnamed: 0                                 0
country_name                               0
country_iso2                               0
week                                       0
country_category                           0
country_weekly_rank                        0
show_title                                 0
season_title                          234434
country_cumulative_weeks_in_top_10         0
global_category                       152263
global_weekly_rank                    152263
global_weekly_hours_viewed            152263
runtime                               277665
global_weekly_views                   277665
global_cumulative_weeks_in_top_10     152263
dtype: int64

In [10]:
duckdb.sql("""
DESCRIBE netflix
""").df()


,column_name,column_type,null,key,default,extra
0,Unnamed: 0,BIGINT,YES,None,None,None
1,country_name,VARCHAR,YES,None,None,None
2,country_iso2,VARCHAR,YES,None,None,None
3,week,VARCHAR,YES,None,None,None
4,country_category,VARCHAR,YES,None,None,None
5,country_weekly_rank,BIGINT,YES,None,None,None
6,show_title,VARCHAR,YES,None,None,None
7,season_title,VARCHAR,YES,None,None,None
8,country_cumulative_weeks_in_top_10,BIGINT,YES,None,None,None
9,global_category,VARCHAR,YES,None,None,None


## Dataset Checks
Validate structure first before analysis

In [11]:
duckdb.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT show_title) AS unique_show_titles,
    COUNT(DISTINCT week) AS unique_weeks,
    COUNT(DISTINCT country_name) AS unique_countries         
FROM netflix
""").df()


,total_rows,unique_show_titles,unique_weeks,unique_countries
0,458338,11060,246,94


Countries count by activity

In [12]:
duckdb.sql("""
    SELECT
    country_name,
    COUNT(*) AS appearances
FROM netflix
GROUP BY country_name
ORDER BY appearances DESC
""").df()


,country_name,appearances
0,Guatemala,4922
1,Dominican Republic,4922
2,Costa Rica,4922
3,Mexico,4922
4,Paraguay,4922
...,...,...
89,Nigeria,4920
90,United States,4920
91,Saudi Arabia,4920
92,Vietnam,4920


Most Persistent Show Title

In [13]:
duckdb.sql("""
    SELECT
    country_name,
    COUNT(*) AS appearances
FROM netflix
GROUP BY country_name
ORDER BY appearances DESC
""").df()


,country_name,appearances
0,Venezuela,4922
1,Mexico,4922
2,Chile,4922
3,Bolivia,4922
4,Dominican Republic,4922
...,...,...
89,Malaysia,4920
90,Saudi Arabia,4920
91,Norway,4920
92,Jordan,4920


Best Performing Titles

In [14]:
duckdb.sql("""
    SELECT
    show_title,
    AVG(country_weekly_rank) AS avg_rank
FROM netflix
GROUP BY show_title
ORDER BY avg_rank ASC
LIMIT 20
""").df()


,show_title,avg_rank
0,The Call,1.000000
1,Boy,1.000000
2,Oi Vita Mia,1.000000
3,Andragogy,1.000000
4,Our Secret Diary,1.000000
5,How To Choose the Right Husband?,1.000000
6,Daddyku Gangster,1.000000
7,War Machine,1.107527
8,ProkuraTOUR'a Rafał Pacześ,1.500000
9,Il sesso degli angeli,1.500000


### UX Team's Choice of Film & TV 
- UX has decided to only use 1 Swedish Film and 1 Swedish TV Series to use for their Dashboard
- Swedish TV Series = Young Royals
- Swedish Films = The Jönsson Gang Returns

In [15]:
duckdb.sql("""
    SELECT *
FROM netflix
WHERE show_title = 'Young Royals'
""").df()

,Unnamed: 0,country_name,country_iso2,week,country_category,country_weekly_rank,show_title,season_title,country_cumulative_weeks_in_top_10,global_category,global_weekly_rank,global_weekly_hours_viewed,runtime,global_weekly_views,global_cumulative_weeks_in_top_10
0,2100,Argentina,AR,2024-03-17,TV,9,Young Royals,Young Royals: Season 3,1,TV (Non-English),5.0,11500000.0,237.0,2900000.0,1.0
1,11922,Austria,AT,2024-03-24,TV,10,Young Royals,Young Royals: Season 3,2,TV (Non-English),9.0,7800000.0,294.0,1600000.0,2.0
2,11937,Austria,AT,2024-03-17,TV,5,Young Royals,Young Royals: Season 3,1,TV (Non-English),5.0,11500000.0,237.0,2900000.0,1.0
3,13360,Austria,AT,2022-11-06,TV,8,Young Royals,Young Royals: Season 2,1,TV (Non-English),3.0,18860000.0,NaN,NaN,1.0
4,31605,Belgium,BE,2024-03-24,TV,10,Young Royals,Young Royals: Season 3,2,TV (Non-English),9.0,7800000.0,294.0,1600000.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
122,425992,Ukraine,UA,2024-03-17,TV,9,Young Royals,Young Royals: Season 1,3,None,NaN,NaN,NaN,NaN,NaN
123,427409,Ukraine,UA,2022-11-06,TV,6,Young Royals,Young Royals: Season 2,1,TV (Non-English),3.0,18860000.0,NaN,NaN,1.0
124,428772,Ukraine,UA,2021-07-18,TV,9,Young Royals,Young Royals: Season 1,2,None,NaN,NaN,NaN,NaN,NaN
125,428787,Ukraine,UA,2021-07-11,TV,4,Young Royals,Young Royals: Season 1,1,TV (Non-English),8.0,9820000.0,NaN,NaN,1.0


In [16]:
duckdb.sql("""
    SELECT *
FROM netflix
WHERE show_title = 'A Day and a Half'
""").df()

,Unnamed: 0,country_name,country_iso2,week,country_category,country_weekly_rank,show_title,season_title,country_cumulative_weeks_in_top_10,global_category,global_weekly_rank,global_weekly_hours_viewed,runtime,global_weekly_views,global_cumulative_weeks_in_top_10
0,2622,Argentina,AR,2023-09-10,Films,1,A Day and a Half,None,2,Films (Non-English),1.0,10400000.0,96.0,6500000.0,2.0
1,2645,Argentina,AR,2023-09-03,Films,4,A Day and a Half,None,1,Films (Non-English),3.0,7100000.0,96.0,4400000.0,1.0
2,17385,Bahamas,BS,2023-09-10,Films,1,A Day and a Half,None,2,Films (Non-English),1.0,10400000.0,96.0,6500000.0,2.0
3,17408,Bahamas,BS,2023-09-03,Films,4,A Day and a Half,None,1,Films (Non-English),3.0,7100000.0,96.0,4400000.0,1.0
4,22313,Bahrain,BH,2023-09-10,Films,9,A Day and a Half,None,1,Films (Non-English),1.0,10400000.0,96.0,6500000.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,416701,Trinidad and Tobago,TT,2023-09-03,Films,10,A Day and a Half,None,1,Films (Non-English),3.0,7100000.0,96.0,4400000.0,1.0
84,446196,Uruguay,UY,2023-09-10,Films,1,A Day and a Half,None,2,Films (Non-English),1.0,10400000.0,96.0,6500000.0,2.0
85,446219,Uruguay,UY,2023-09-03,Films,4,A Day and a Half,None,1,Films (Non-English),3.0,7100000.0,96.0,4400000.0,1.0
86,451119,Venezuela,VE,2023-09-10,Films,2,A Day and a Half,None,2,Films (Non-English),1.0,10400000.0,96.0,6500000.0,2.0


# Netflix EDA | show title centric 

## Top Performers
What are the Top 10 titles per category per year?

In [17]:
duckdb.sql("""
WITH base AS (
    SELECT 
        DATE_TRUNC('year', STRPTIME(week, '%Y-%m-%d')) AS year,
        global_category,
        show_title,
        COUNT(*) AS appearances,
        AVG(country_weekly_rank) AS avg_rank,
        MAX(country_cumulative_weeks_in_top_10) AS longevity
    FROM netflix
    GROUP BY 
        year,
        global_category,
        show_title
),

ranked AS (
    SELECT *,
        ROW_NUMBER() OVER (
            PARTITION BY year, global_category
            ORDER BY 
                appearances DESC,        -- primary KPI
                avg_rank ASC             -- tie-breaker (lower is better)
        ) AS rank_in_category_year
    FROM base
)

SELECT *
FROM ranked
WHERE rank_in_category_year
ORDER BY year, global_category, rank_in_category_year
""").df()

,year,global_category,show_title,appearances,avg_rank,longevity,rank_in_category_year
0,2021-01-01,Films (English),Red Notice,581,3.399312,7,1
1,2021-01-01,Films (English),Army of Thieves,336,2.964286,4,2
2,2021-01-01,Films (English),Love Hard,296,4.195946,5,3
3,2021-01-01,Films (English),Kate,287,3.233449,4,4
4,2021-01-01,Films (English),The Guilty,285,2.343860,4,5
...,...,...,...,...,...,...,...
19357,2026-01-01,None,City of Shadows,1,10.000000,5,987
19358,2026-01-01,None,Mamma Mia! Here We Go Again,1,10.000000,1,988
19359,2026-01-01,None,Sounds of Winter,1,10.000000,1,989
19360,2026-01-01,None,Train Dreams,1,10.000000,3,990
